In [1]:
import pandas as pd
import numpy as np
import spacy

nlp = spacy.load('en_core_web_sm') # english model
df = pd.read_csv('../data/processed/combined_en.csv')

print(df.shape)
print(df['label'].value_counts())
df.head()

(9799, 2)
label
emotional_manipulation    5390
rational_argument         3522
demagogy_tricks            887
Name: count, dtype: int64


,text,label
0,Proof The Mainstream Media Is Manipulating The...,emotional_manipulation
1,Charity Clinton Foundation Distributed Watered...,emotional_manipulation
2,A Hillary Clinton Administration May be Entire...,emotional_manipulation
3,Trumps Latest Campaign Promise May Be His Most...,emotional_manipulation
4,Website is Down For Maintenance. Website is Do...,emotional_manipulation


In [2]:
sample = df['text'].iloc[0]
print(f'text, {sample[:200]}')

doc = nlp(sample)

for token in list(doc)[:15]:
    print(f'{token.text:20} POS: {token.pos_:10} DEP: {token.dep_}')

text, Proof The Mainstream Media Is Manipulating The Election By Taking Bill Clinton Out Of Context. I woke up this morning to find a variation of this headline splashed all over my news feed Bill Clinton N
Proof                POS: PROPN      DEP: npadvmod
The                  POS: DET        DEP: det
Mainstream           POS: PROPN      DEP: compound
Media                POS: PROPN      DEP: nsubj
Is                   POS: AUX        DEP: aux
Manipulating         POS: VERB       DEP: ROOT
The                  POS: DET        DEP: det
Election             POS: PROPN      DEP: dobj
By                   POS: ADP        DEP: prep
Taking               POS: VERB       DEP: pcomp
Bill                 POS: PROPN      DEP: compound
Clinton              POS: PROPN      DEP: dobj
Out                  POS: ADV        DEP: advmod
Of                   POS: ADP        DEP: prep
Context              POS: PROPN      DEP: pobj


In [3]:
def extract_features(text):
    doc = nlp(text)
    tokens = [t for t in doc if not t.is_space]
    sentences = list(doc.sents)
    n_tokens = len(tokens) + 1

    we_words = {'we', 'our', 'us'}
    they_words = {'they', 'them', 'their', 'enemy', 'enemies'}
    we_ratio = sum(1 for t in tokens if t.lemma_.lower() in we_words) / n_tokens
    they_ratio = sum(1 for t in tokens if t.lemma_.lower() in they_words) / n_tokens

    exclaim_ratio = sum(1 for s in sentences if '!' in s.text) / (len(sentences) + 1)

    question_ratio = sum(1 for s in sentences if '?' in s.text) / (len(sentences) + 1)

    modal_words = {'must', 'should', 'need', 'have to', 'ought'}
    modal_ratio = sum(1 for t in tokens if t.lemma_.lower() in modal_words) / n_tokens

    logic_words = ['because', 'therefore', 'thus', 'however', 'consequently', 'moreover']
    logic_count = sum(1 for lw in logic_words if lw in text.lower())

    adj_ratio = sum(1 for t in tokens if t.pos_ == 'ADJ') / n_tokens

    verb_ratio = sum(1 for t in tokens if t.pos_ == 'VERB') / n_tokens

    avg_sent_len = sum(len(list(s)) for s in sentences) / (len(sentences) + 1)

    caps_ratio = sum(1 for t in tokens if t.text.isupper() and len(t.text) > 1) / n_tokens

    return {
        'we_ratio': we_ratio,
        'they_ratio': they_ratio,
        'exclaim_ratio': exclaim_ratio,
        'question_ratio': question_ratio,
        'modal_ratio': modal_ratio,
        'logic_count': logic_count,
        'adj_ratio': adj_ratio,
        'verb_ratio': verb_ratio,
        'avg_sent_len': avg_sent_len,
        'caps_ratio': caps_ratio
    }

result = extract_features(df['text'].iloc[0])
for k,v in sorted(result.items(), key=lambda x: x[1], reverse=True):
    print(f'{k:20}{v:.6f}')

avg_sent_len        25.291667
logic_count         1.000000
verb_ratio          0.130148
question_ratio      0.041667
adj_ratio           0.041186
they_ratio          0.011532
caps_ratio          0.004942
we_ratio            0.001647
modal_ratio         0.001647
exclaim_ratio       0.000000


In [4]:
from tqdm import tqdm
tqdm.pandas()

features_df = df['text'].progress_apply(extract_features)
features_df = pd.DataFrame(features_df.tolist())

features_df['label'] = df['label'].values

print(features_df.shape)
features_df.head()

100%|██████████| 9799/9799 [01:53<00:00, 86.70it/s] 

(9799, 11)


,we_ratio,they_ratio,exclaim_ratio,question_ratio,modal_ratio,logic_count,adj_ratio,verb_ratio,avg_sent_len,caps_ratio,label
0,0.001647,0.011532,0.0,0.041667,0.001647,1,0.041186,0.130148,25.291667,0.004942,emotional_manipulation
1,0.000000,0.007412,0.0,0.000000,0.000000,1,0.072776,0.114555,25.169492,0.028302,emotional_manipulation
2,0.012030,0.001504,0.0,0.166667,0.007519,0,0.079699,0.093233,27.708333,0.003008,emotional_manipulation
3,0.012077,0.028986,0.0,0.131579,0.004831,1,0.068841,0.125604,21.921053,0.004831,emotional_manipulation
4,0.000000,0.000000,0.0,0.000000,0.000000,0,0.166667,0.000000,3.666667,0.000000,emotional_manipulation


In [5]:
mean_by_class = features_df.groupby('label').mean().round(4)
print(mean_by_class.T)

label           demagogy_tricks  emotional_manipulation  rational_argument
we_ratio                 0.0055                  0.0079             0.0085
they_ratio               0.0059                  0.0051             0.0048
exclaim_ratio            0.0080                  0.0031             0.0020
question_ratio           0.0087                  0.0031             0.0033
modal_ratio              0.0011                  0.0010             0.0010
logic_count              0.1116                  0.0419             0.0400
adj_ratio                0.0495                  0.0562             0.0635
verb_ratio               0.1213                  0.1150             0.1040
avg_sent_len            10.7837                  9.8960             9.9448
caps_ratio               0.0080                  0.0067             0.0066


In [6]:
import os
os.makedirs('../data/processed', exist_ok=True)

features_df.to_csv('../data/processed/features_en.csv', index=False)
print(f'size: {features_df.shape}')

size: (9799, 11)


In [7]:
import sys
sys.path.append('..')
from src.features.extractor import extract_features as extract_features_src

print(extract_features_src(df["text"].iloc[0]))

{'we_ratio': 0.0016474464579901153, 'they_ratio': 0.011532125205930808, 'exclaim_ratio': 0.0, 'question_ratio': 0.041666666666666664, 'modal_ratio': 0.0016474464579901153, 'logic_count': 1, 'adj_ratio': 0.04118616144975288, 'verb_ratio': 0.1301482701812191, 'avg_sent_len': 25.291666666666668, 'caps_ratio': 0.004942339373970346}
